In [4]:
from googleapiclient import discovery
service = discovery.build('healthcare', 'v1')

/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.21). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its e

In [8]:
from googleapiclient import discovery
from google.auth import default

def analyze_medical_text(project_id, location, text_content):
    credentials, _ = default()
    service = discovery.build('healthcare', 'v1', credentials=credentials)

    nlp_service_name = f"projects/{project_id}/locations/{location}/services/nlp"
    body = {"documentContent": text_content}

    response = service.projects().locations().services().nlp().analyzeEntities(
        nlpService=nlp_service_name,
        body=body
    ).execute()

    entities = response.get("entities", [])
    print(f"Found {len(entities)} medical entities\n")
    print("=" * 70)

    for entity in entities:
        entity_id = entity.get("entityId", "N/A")
        preferred_term = entity.get("preferredTerm", "N/A")
        vocab_codes = entity.get("vocabularyCodes", [])

        print(f"\n ENTITY: {preferred_term}")
        print(f"  ID         : {entity_id}")

        # Additional vocab codes (ICD-10, SNOMED, RxNorm etc.)
        if vocab_codes:
            print(f"  Codes      : {', '.join(vocab_codes)}")

        mentions = entity.get("mentions", [])
        print(f"  Mentions   : {len(mentions)}")

        for i, mention in enumerate(mentions, 1):
            text_obj = mention.get("text", {})
            surface_text = text_obj.get("content", "N/A")
            begin_offset = text_obj.get("beginOffset", "N/A")
            mention_type = mention.get("type", "N/A")
            certainty = mention.get("certaintyAssessment", {})
            confidence = certainty.get("confidence", None)
            certainty_value = certainty.get("value", "N/A")
            subject = mention.get("subject", {}).get("value", "N/A")
            temporal = mention.get("temporalAssessment", {}).get("value", "N/A")

            print(f"\n    [{i}] \"{surface_text}\"")
            print(f"        Type        : {mention_type}")
            print(f"        Certainty   : {certainty_value}", end="")
            if confidence is not None:
                print(f"  (confidence: {confidence:.2%})", end="")
            print()
            print(f"        Subject     : {subject}")
            print(f"        Temporal    : {temporal}")
            print(f"        Offset      : char {begin_offset}")

        print("-" * 70)

PROJECT_ID = "bio-ramp-ner"
LOCATION = "us-central1"
TEXT = "Hello, good evening. Hello, good evening. Welcome to clinic. I'm Dr. Dina. Yes, Doctor. I am Tola, 35 years old, female. Sorry, how old? I didn't get that. 35. 35. Okay. You're welcome, Tola. Yes, doctor. Okay, so what brings you to the clinic today? Doctor, I've been having this problem now for the last two weeks. Sorry to hear that. Where is the pain located in your abdomen the left the right no it's that upper part of the central and you don't want the left or right is that middle part the middle part that is closer to your chest or your breast yes it's closer to yes it's that place okay sorry to hear that and you said this has been ongoing for how long for the last two weeks oh doctor that would have been so so discomforting sorry to hear that yes doctor so what is this pain like can? Doctor, this pain is like it's burning me. It's like it's burning. Sorry. Does it come and go or is it always there? Yes, it comes and goes. It will come, it will go. Okay. Is there anything that makes this pain worse? Doctor, once I refuse to eat like this, the pain will be so much. Okay. When you eat, does it feel better? Yes. Once I eat food like this, everything else stops and reduces by itself. Okay. Are there things that you eat that make this pain also bad, even though you just eat? I can't seem to eat anything. I can't seem. There's nothing. Once I eat, the pain just reduces. Oh. Sorry to hear that. Are you vomiting? No, I'm not vomiting, doctor. Are you purging, like going to the toilet frequently? No, doctor. I'm not going to the toilet frequently. Any temperature? Doctor, my body is not... Okay. This pain, do you feel it any other place apart from that middle of your chest, like maybe your back, your shoulder? Yes, that pain usually goes to my back when it starts okay uh if i could uh run this pain on a scale zero to ten where zero is like where you don't have any pain and ten the worst pain of your life where would this pain be yeah doctor Doctor, the pain is usually between 3 and 6, so sometimes 3, sometimes 6. That's much."

analyze_medical_text(PROJECT_ID, LOCATION, TEXT)

Found 19 medical entities


 ENTITY: Abdomen
  ID         : UMLS/C0000726
  Codes      : FMA/9577, LNC/LA12706-0, LNC/LA4238-7, LNC/LP199956-6, LNC/LP234820-1, LNC/LP6990-8, LNC/MTHU001425, LNC/MTHU059270, MSH/D000005, MTH/NOCODE, OMIM/MTHU000219
  Mentions   : 0
----------------------------------------------------------------------

 ENTITY: Back
  ID         : UMLS/C0004600
  Codes      : FMA/25056, FMA/71938, LNC/LA12705-2, LNC/LP7046-8, LNC/MTHU001427, MSH/D001415, MTH/NOCODE, NCI/C32481
  Mentions   : 0
----------------------------------------------------------------------

 ENTITY: Breast
  ID         : UMLS/C0006141
  Codes      : FMA/9601, LNC/LA4255-1, LNC/LP199961-6, LNC/LP29849-4, LNC/LP7086-4, LNC/MTHU001046, LNC/MTHU011382, MSH/D001940, MTH/U001764, NCI/C12971, NCI/C43595, OMIM/MTHU001433
  Mentions   : 0
----------------------------------------------------------------------

 ENTITY: Burn injury
  ID         : UMLS/C0006434
  Codes      : ICD10CM/T30.0, ICD9CM/940-949.99,